# 📊 Vox-SN — Notebook d'Exploration

**Objectif** : explorer interactivement les posts ingérés via Kafka et stockés dans Hive.

**Prérequis** : services Docker démarrés (`make up`) + données générées (`python scripts/seed_training_data.py`).

---

## 1. Setup & Imports

In [ ]:
import sys
import os

# Ajouter le projet au PYTHONPATH
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../spark'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print('✅ Imports OK')

## 2. Charger les données seed

In [ ]:
# Lecture du CSV généré par scripts/seed_training_data.py
df = pd.read_csv('../data/samples/training_data.csv', parse_dates=['date_post'])
print(f'Total posts : {len(df)}')
df.head()

In [ ]:
df.info()
print('\nValeurs manquantes :', df.isnull().sum().sum())

## 3. Distribution par service

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['service_cible'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Nombre de posts par service')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

df.groupby('service_cible')['sentiment_score'].mean().sort_values().plot(
    kind='barh', ax=axes[1], 
    color=['#C72E29' if v < 0 else '#28A745' for v in df.groupby('service_cible')['sentiment_score'].mean().sort_values()]
)
axes[1].set_title('Sentiment moyen par service')
axes[1].axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

## 4. Battle Mobile Money

In [ ]:
mm = df[df['service_cible'].isin(['WAVE', 'ORANGE_MONEY', 'FREE_MONEY'])]

battle = mm.groupby('service_cible').agg(
    nb_mentions=('post_id', 'count'),
    sentiment_moyen=('sentiment_score', 'mean'),
    pct_positif=('sentiment_label', lambda x: (x == 'POSITIF').sum() / len(x) * 100),
    pct_critique=('sentiment_label', lambda x: (x == 'NEGATIF_FORT').sum() / len(x) * 100),
    nb_fraudes=('categorie', lambda x: (x == 'FRAUDE').sum())
).round(3)

print('⚔️ Battle Mobile Money :')
battle

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

battle['sentiment_moyen'].plot(kind='bar', ax=axes[0], color=['#06b6d4', '#f97316', '#a855f7'])
axes[0].set_title('Sentiment moyen')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].tick_params(axis='x', rotation=45)

battle['pct_critique'].plot(kind='bar', ax=axes[1], color='#C72E29')
axes[1].set_title('% posts critiques')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=45)

battle['nb_fraudes'].plot(kind='bar', ax=axes[2], color='#dc2626')
axes[2].set_title('Nb plaintes Fraude')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Catégories de plaintes

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ct = pd.crosstab(df['service_cible'], df['categorie'])
ct.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
ax.set_title('Catégorisation des plaintes par service')
ax.set_xlabel('')
ax.legend(title='Catégorie', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

## 6. WordCloud des plaintes Wave

In [ ]:
wave_textes = ' '.join(df.loc[
    (df['service_cible'] == 'WAVE') & (df['sentiment_score'] < 0),
    'texte_clean'
].dropna().tolist())

if wave_textes.strip():
    wc = WordCloud(width=900, height=400, background_color='white',
                   colormap='Reds', max_words=80).generate(wave_textes)
    plt.figure(figsize=(14, 6))
    plt.imshow(wc)
    plt.axis('off')
    plt.title('Mots-clés des plaintes Wave (Wolof + Français)')
    plt.show()
else:
    print('⚠️ Pas assez de données pour générer le wordcloud')

## 7. Répartition par langue

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['langue'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%',
                                 colors=['#3b82f6', '#f97316', '#10b981'])
axes[0].set_title('Répartition des langues')
axes[0].set_ylabel('')

df.groupby('langue')['sentiment_score'].mean().plot(
    kind='bar', ax=axes[1], color=['#3b82f6', '#10b981', '#f97316']
)
axes[1].set_title('Sentiment moyen par langue')
axes[1].axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

## 8. Évolution temporelle

In [ ]:
trend = df.groupby(['date_post', 'service_cible'])['sentiment_score'].mean().unstack()

fig, ax = plt.subplots(figsize=(14, 6))
for col in trend.columns:
    ax.plot(trend.index, trend[col], label=col, marker='o', markersize=3)
ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(-0.5, color='red', linestyle='--', alpha=0.5, label='Seuil crise')
ax.set_title('Évolution du sentiment moyen sur 30 jours')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xlabel('Date')
ax.set_ylabel('Sentiment')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Conclusion

Insights extraits :

- **Wave** : leader en part de voix mais avec des pics de plaintes TECHNIQUE
- **Senelec** : sentiment le plus négatif, dominé par catégorie TECHNIQUE (pannes)
- **TER** : meilleur sentiment global, peu de mentions critiques
- **Posts en Wolof** : représentent ~30% du volume, ne peuvent être ignorés

Ces insights guident les opérateurs pour prioriser leurs investissements (support client, infrastructure).

---

*Vox-SN © 2025-2026 — UADB M2 BD&IA*